# Self-attractive chemotaxis

Packages used:

In [ ]:
using Pkg
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
using CSV, DataFrames, Statistics
using Printf, JLD2
using SpecialFunctions
using LsqFit
using LinearAlgebra
using StatsBase
using Images, FileIO
using Graphs

## Model

The following code generates the model, with the rules for the agents, and defines the medium.

In [ ]:
self_positive = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64, 
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :active => Bool,

        :S => Float64,

        :methyl => Float64,
        :Yp => Float64, 
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64,
        :M => Float64
        
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64, 
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, 
        :Ki => Float64,
        :Ka => Float64,
        :τm => Float64, 
        :α => Float64,  
        :ωFrec => Float64,    
        :Ky => Float64,    
        :Z => Float64,         
        :Kz => Float64,         
        :Yy => Float64,        

        :DMedium => Float64,
        :delta => Float64
    ),

    medium = Dict(
        :mm => Float64
    ),

    agentODE = quote

        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]


        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)
        

        if x < xmin
            idx = Int(floor(Int, (x+(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        elseif x > xmax
            idx = Int(floor(Int, (x-(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        end

        if y < ymin
            idy = Int(floor(Int,(y+ymax-ymin)/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)

        elseif y > ymax
            idy = Int(floor(Int,(y-(ymax-ymin))/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)
    
        end
        
        mmb = max(0, mm[idx,idy])

        F = ε0 + ε1 * methyl + Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka)) 
  
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)       

        mx = (ε0 + Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka)) - F0) / (- ε1)

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))     

        dt(x) = vx  
        dt(y) = vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)       
        
    end,

    agentRule = quote
        
            xmin, xmax = simBox[1,1], simBox[1,2]
            ymin, ymax = simBox[2,1], simBox[2,2]

            idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
            idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2) 

            v_run = v
            v_tumble = 0.25 

            speed = active ? v_run : v_tumble

            Dr_tumble = 6.2      
            Dr_total = active ? Dr_run : Dr_tumble

            mm[idx,idy] += S

            if active 
                λ = ωFrec*exp(-G)
                P = 1 - exp(-λ * dt)
                
            else
                λ = ωFrec*exp(G)
                P = 1 - exp(-λ * dt)  
                
            end


            if active 
                λrt = ωFrec*exp(-G) 

                P_rt = 1 - exp(-λrt * dt)
                P = rand() 
                                                   
                if P < P_rt       
                    active = false
                    vx = speed* cos(theta) 
                    vy = speed* sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()       
                else   
                    active = true
                    vx = speed * cos(theta) 
                    vy = speed * sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()      
                end

            else
                λtr = ωFrec*exp(G) 
                P_tr = 1 - exp(-λtr * dt)
                P = rand()

                if P < P_tr
                    active = true
                    vx = speed * cos(theta) 
                    vy = speed * sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()
                else
                    active = false
                    vx = speed* cos(theta) 
                    vy = speed* sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()
                end
            end

            
            xmin, xmax = simBox[1,1], simBox[1,2]
            ymin, ymax = simBox[2,1], simBox[2,2]

            # ================
            #  Boundry conditions
            # ================

            if x < xmin
                x += (xmax - xmin)
            elseif x > xmax
                x -= (xmax - xmin)
            end

            if y < ymin
                y += (ymax - ymin)
            elseif y > ymax
                y -= (ymax - ymin)
            end


    end,


    mediumODE = quote
        if @mediumInside()
            dt(mm) = DMedium *(@∂2(1, mm)+ @∂2(2, mm)) - delta*mm
        elseif @mediumBorder(1,-1)
            mm = mm[NMedium[1] - 1, i2_]
        elseif @mediumBorder(1,1)
            mm = mm[1, i2_]
        elseif @mediumBorder(2,1)
            mm = mm[i1_, 1]
        elseif @mediumBorder(2,-1)
            mm = mm[i1_, NMedium[2] - 1]
        end
    end,


    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Heun(),
    neighborsAlg=CBMNeighbors.CellLinked(cellEdge=10)
)

Next, the following code loads the communities and evolves them for the parameteres presented in the thesis.

In [ ]:
Dc_vals = [40, 13.33, 13.33, 2.0, 1.25]
d_vals = [0.0025, 0.0075, 0.03, 0.0125, 0.08]

for r in 1:3
    for (idx, n) in enumerate(Dc_val)
        for (idx_t, tm) in enumerate(tms)

            com = Community(
                collective_PBC,
                N=500,
                dt=0.01,
                simBox = [-100.0 150.0; -100.0 150.0],
                NMedium = [250, 250]
            )

            m = 1/100
            g = 1/10000
            d = 1

            com.Dr_run = 0.062

            com.v = vls[idx_t]

            com.DMedium = Dc_val[idx]
            com.delta = delta_val[idx]

            com.ωFrec = 1.3
            com.Ki = 0.0182
            com.Ka = 3.0
            com.Nrec = 6.0
            com.ε0   = 6.0
            com.ε1   = -1.0
            com.ε2   = 80
            com.ε3   = 80

            com.τm = tm

            com.α   = 6.0

            com.K = 2.0 

            com.Ky = 100.0
            com.Kz = 10.0
            com.Z = 5.0
            com.Yy = 0.1

            com.m = 1.        
            com.d = 1.        
            com.l = 3;

            com.x = rand(Uniform(com.simBox[1,:]...),com.N)
            com.y = rand(Uniform(com.simBox[2,:]...),com.N)
            com.theta = rand(Uniform(0,2π),com.N)

            com.methyl .= 0.0
            com.Yp .= com.K


            com.active .= true 

            com.S = 0.0025
            com.ve = 0

            outfile = "cluster_v$(vls[idx_t])_tm_$(tm)_l$(ls[idx])_$(r).jld2"
            steps = 50000

            loadToPlatform!(com, preallocateAgents=500)
            com.mm = zeros(Float64, com.NMedium...)

            jldopen(outfile, "w") do file

                for step in 1:steps
                    step!(com)
                    if step % 100 == 0
                        stepname = @sprintf("step_%06d", step)
                        g = JLD2.Group(file, stepname)

                        # Agent-level arrays (length = N)
                        g["x"] = copy(com.x)
                        g["y"] = copy(com.y)
                        g["theta"] = copy(com.theta)
                        g["mm_grid"] = copy(com.mm)
                        if step%5000 == 0
                            println("l = $(ls[idx]), tm = $tm, step = $step")
                        end
                    end
                end
            end
        end
    end
end



## Analysis

The following code extracts the last time step concentration map to be calculated with ImageJ:

In [ ]:
data = Dict{Int, Any}()
step = 50000
tms = [1.0, 2.0, 4.0, 10.0]
vel = [10.0, 7.07, 5.0, 3.16]
ls = [4]
for i in 1:3
    for (idx, tm) in enumerate(tms)
        for (idx_l, l) in enumerate(ls)
            jldopen("cluster_v$(vel[idx])_tm_$(tm)_l$(l)_$i.jld2", "r") do file
                    key = file[@sprintf("step_%06d", step)]
                    data[step] = Dict(
                        "mm_grid" => copy(key["mm_grid"]))
            end
            grid = data[step]["mm_grid"]

            A_wrap = [
                grid  grid  grid;
                grid  grid  grid;
                grid  grid  grid
            ]
            # Convert to image object
            img = Gray.(A_wrap)

            # Save as TIFF
            save("Fiji_images/v$(vel[idx])_tm_$(tm)_l$(l)_$(i).tif", img)
        end
    end
end

Then, the time to clustering was evaluated as follows:

In [ ]:
# ============================================================
# PARAMETERS
# ============================================================

steps = 50000
saved_steps = 100:100:steps

r_cluster = 5.0    

parameter_sets = [
    (v = 10.0, tm = 1.0),
    (v = 7.07, tm = 2.0),
    (v = 5.0, tm = 4.0),
    (v = 3.16, tm = 10.0)
]

# ============================================================
# OUTPUT DATAFRAME
# ============================================================

results = DataFrame(
    v = Float64[],
    tm = Float64[],
    set = Int[],
    t_cluster = Float64[]
)

# ============================================================
# LOOP THROUGH ALL SIMULATIONS
# ============================================================

for pars in parameter_sets

    v = pars.v
    tm = pars.tm

    for rep in 1:n_repeats

        filename = "cluster_v$(v)_tm_$(tm)_l42_$(rep).jld2"

        println("Processing ", filename)

        largest_fraction = Float64[]
        times = Float64[]

        jldopen(filename, "r") do file

            for step in saved_steps

                g = file[@sprintf("step_%06d", step)]

                x = g["x"]
                y = g["y"]

                # ------------------------------------------------
                # Remove inactive agents
                # ------------------------------------------------

                valid = (x .!= 0) .| (y .!= 0)

                x = x[valid]
                y = y[valid]

                N = length(x)

                # Skip empty snapshots just in case
                if N == 0
                    continue
                end

                # ------------------------------------------------
                # Build contact graph
                # ------------------------------------------------

                G = SimpleGraph(N)

                for i in 1:(N-1)
                    for j in (i+1):N

                        dx = x[i] - x[j]
                        dy = y[i] - y[j]

                        if dx^2 + dy^2 < r_cluster^2
                            add_edge!(G, i, j)
                        end
                    end
                end

                # ------------------------------------------------
                # Largest cluster fraction
                # ------------------------------------------------

                comps = connected_components(G)

                cluster_sizes = length.(comps)

                push!(
                    largest_fraction,
                    maximum(cluster_sizes) / N
                )

                push!(
                    times,
                    step * 0.01
                )
            end
        end

        # --------------------------------------------------------
        # Compute clustering time
        # --------------------------------------------------------

        final_fraction = largest_fraction[end]

        threshold = 0.95 * final_fraction

        idx = findfirst(x -> x ≥ threshold, largest_fraction)

        t_cluster = isnothing(idx) ? NaN : times[idx]

        println("   t_cluster = ", t_cluster)

        push!(
            results,
            (
                v = v,
                tm = tm,
                set = rep,
                t_cluster = t_cluster
            )
        )
    end
end